In [2]:


import re
import numpy as np
from pathlib import Path

# ── Paste your raw data here ──────────────────────────────────────────────────
raw = """
Function 1:	[0.004403, 0.000331]
Function 2:	[0.002254, 0.997016]
Function 3:	[0.424138, 0.996905, 0.903360]
Function 4:	[0.397651, 0.472910, 0.385315, 0.418103]
Function 5:	[0.999999, 0.999999, 0.880310, 0.999999]
Function 6:	[0.446564, 0.418053, 0.499937, 0.999999, 0.000001]
Function 7:	[0.000001, 0.081653, 0.000001, 0.297573, 0.268360, 0.733459]
Function 8:	[0.210472, 0.196953, 0.208807, 0.293054, 0.999999, 0.908902, 0.211589, 0.406913]

Function 1:	1.8795868137605964e-245
Function 2:	0.02067653460932504
Function 3:	-0.11879595583550381
Function 4:	-1.1208424476469427
Function 5:	6830.979695854196
Function 6:	-0.5780173468025228
Function 7:	1.9122454781778415
Function 8:	9.7430936991566
"""

# ── Parse ─────────────────────────────────────────────────────────────────────
def parse_submission(text):
    new_inputs = {}
    new_outputs = {}

    for line in text.strip().splitlines():
        line = line.strip()
        if not line:
            continue

        m = re.match(r'Function\s+(\d+):\s+\[([^\]]+)\]', line)
        if m:
            fn = int(m.group(1))
            new_inputs[fn] = [float(v.strip()) for v in m.group(2).split(',')]
            continue

        m = re.match(r'Function\s+(\d+):\s+([-\d.e+]+)', line)
        if m:
            fn = int(m.group(1))
            new_outputs[fn] = float(m.group(2))

    return new_inputs, new_outputs


new_inputs, new_outputs = parse_submission(raw)

print("new_inputs = {")
for k, v in sorted(new_inputs.items()):
    print(f"    {k}: {v},")
print("}\n")

print("new_outputs = {")
for k, v in sorted(new_outputs.items()):
    print(f"    {k}: {v},")
print("}\n")

# ── Append to .npy files ──────────────────────────────────────────────────────
base = Path("initial_data_7")

for fn in range(1, 9):
    folder = base / f"function_{fn}"
    inp_path = folder / "initial_inputs.npy"
    out_path = folder / "initial_outputs.npy"

    X = np.load(inp_path)
    Y = np.load(out_path)

    print(f"Function {fn}:")
    print(f"  Inputs  before: {X.shape}  -> ", end="")

    x_new = np.array(new_inputs[fn]).reshape(1, -1)
    y_new = np.array([new_outputs[fn]])

    X_updated = np.vstack([X, x_new])
    Y_updated = np.append(Y, y_new)

    print(f"{X_updated.shape}")
    print(f"  Outputs before: {Y.shape}  -> {Y_updated.shape}")
    print(f"  New x: {x_new.ravel()}")
    print(f"  New y: {y_new[0]:.6e}")

    np.save(inp_path, X_updated)
    np.save(out_path, Y_updated)

print("\nDone — all files updated.")

new_inputs = {
    1: [0.004403, 0.000331],
    2: [0.002254, 0.997016],
    3: [0.424138, 0.996905, 0.90336],
    4: [0.397651, 0.47291, 0.385315, 0.418103],
    5: [0.999999, 0.999999, 0.88031, 0.999999],
    6: [0.446564, 0.418053, 0.499937, 0.999999, 1e-06],
    7: [1e-06, 0.081653, 1e-06, 0.297573, 0.26836, 0.733459],
    8: [0.210472, 0.196953, 0.208807, 0.293054, 0.999999, 0.908902, 0.211589, 0.406913],
}

new_outputs = {
    1: 1.8795868137605964e-245,
    2: 0.02067653460932504,
    3: -0.11879595583550381,
    4: -1.1208424476469427,
    5: 6830.979695854196,
    6: -0.5780173468025228,
    7: 1.9122454781778415,
    8: 9.7430936991566,
}

Function 1:
  Inputs  before: (15, 2)  -> (16, 2)
  Outputs before: (15,)  -> (16,)
  New x: [0.004403 0.000331]
  New y: 1.879587e-245
Function 2:
  Inputs  before: (15, 2)  -> (16, 2)
  Outputs before: (15,)  -> (16,)
  New x: [0.002254 0.997016]
  New y: 2.067653e-02
Function 3:
  Inputs  before: (20, 3)  -> (21, 3)
  Outputs before: (20